In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy qiskit-ibm-catalog scikit-learn
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Singularity Machine Learning - Clasificación: una Qiskit Function de Multiverse Computing
*Consulta la [referencia de API](https://docs.quantum.ibm.com/api/functions/multiverse-computing-singularity)*



<Accordion>
<AccordionItem title="Versiones de paquetes">

El código de esta página fue desarrollado usando los siguientes requisitos.
Recomendamos usar estas versiones o más recientes.

```
scikit-learn~=1.8.0
```
</AccordionItem>
</Accordion>

> **Note:** * Las Qiskit Functions son una función experimental disponible únicamente para usuarios de los planes IBM Quantum&reg; Premium, Flex y On-Prem (a través de la API de IBM Quantum Platform). Se encuentran en estado de versión preliminar y pueden cambiar.

## Descripción general
Con la función "Singularity Machine Learning - Clasificación" puedes resolver problemas reales de aprendizaje automático en hardware cuántico sin necesitar experiencia en computación cuántica. Esta función de aplicación, basada en métodos ensemble, es un clasificador híbrido. Aprovecha métodos clásicos como boosting, bagging y stacking para el entrenamiento inicial del ensemble. Posteriormente, se emplean algoritmos cuánticos como el solucionador cuántico variacional de valores propios (VQE) y el algoritmo cuántico de optimización aproximada (QAOA) para mejorar la diversidad, las capacidades de generalización y la complejidad global del ensemble entrenado.

A diferencia de otras soluciones de aprendizaje automático cuántico, esta función es capaz de manejar conjuntos de datos a gran escala con millones de ejemplos y características sin estar limitada por el número de qubits del QPU objetivo. El número de qubits solo determina el tamaño del ensemble que se puede entrenar. También es altamente flexible y puede utilizarse para resolver problemas de clasificación en una amplia variedad de dominios, incluyendo finanzas, salud y ciberseguridad.
Logra de manera consistente altas precisiones en problemas clásicamente difíciles que involucran conjuntos de datos de alta dimensión, ruidosos y desbalanceados.

![Cómo funciona](../docs/images/guides/multiverse-computing-singularity/how_it_works.avif)

Está diseñada para:
1. Ingenieros y científicos de datos en empresas que buscan mejorar sus ofertas tecnológicas integrando el aprendizaje automático cuántico en sus productos y servicios,
2. Investigadores en laboratorios de investigación cuántica que exploran aplicaciones de aprendizaje automático cuántico y buscan aprovechar la computación cuántica para tareas de clasificación, y
3. Estudiantes y docentes en instituciones educativas en cursos como aprendizaje automático, que buscan demostrar las ventajas de la computación cuántica.

El siguiente ejemplo muestra sus diversas funcionalidades, incluyendo `create`, `list`, `fit` y `predict`, y demuestra su uso en un problema sintético compuesto por dos semiciclos entrelazados, un problema notoriamente difícil debido a su frontera de decisión no lineal.



## Descripción de la función
Esta Qiskit Function permite a los usuarios resolver problemas de clasificación binaria utilizando el clasificador ensemble mejorado cuánticamente de Singularity. En segundo plano, utiliza un enfoque híbrido para entrenar clásicamente un ensemble de clasificadores sobre el conjunto de datos etiquetado y, a continuación, optimizarlo para maximizar la diversidad y la generalización mediante el Algoritmo Cuántico de Optimización Aproximada (QAOA) en QPUs de IBM&reg;. A través de una interfaz fácil de usar, los usuarios pueden configurar un clasificador según sus requisitos, entrenarlo con el conjunto de datos de su elección y utilizarlo para hacer predicciones sobre un conjunto de datos no visto previamente.

Para resolver un problema de clasificación genérico:
1. Preprocesa el conjunto de datos y divídelo en conjuntos de entrenamiento y prueba. Opcionalmente, puedes dividir además el conjunto de entrenamiento en conjuntos de entrenamiento y validación. Esto se puede lograr usando [scikit-learn](https://scikit-learn.org/1.5/modules/generated/sklearn.model_selection.train_test_split.html).
2. Si el conjunto de entrenamiento está desbalanceado, puedes remuestrearlo para equilibrar las clases usando [imbalanced-learn](https://imbalanced-learn.org/stable/introduction.html#problem-statement-regarding-imbalanced-data-sets).
3. Sube los conjuntos de entrenamiento, validación y prueba por separado al almacenamiento de la función usando el método `file_upload` del catálogo, pasándole la ruta correspondiente cada vez.
4. Inicializa el clasificador cuántico usando la acción `create` de la función, que acepta hiperparámetros como el número y los tipos de aprendices, la regularización (valor lambda) y las opciones de optimización incluyendo el número de capas, el tipo de optimizador clásico, el backend cuántico, entre otros.
5. Entrena el clasificador cuántico sobre el conjunto de entrenamiento usando la acción `fit` de la función, pasándole el conjunto de entrenamiento etiquetado y el conjunto de validación si corresponde.
6. Realiza predicciones sobre el conjunto de prueba no visto previamente usando la acción `predict` de la función.

---

## Primeros pasos
Autentícate usando tu [clave de API de IBM Quantum Platform](http://quantum.cloud.ibm.com/) y selecciona la Qiskit Function de la siguiente manera:

In [6]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Verify that you have access to the function
catalog.list()

[QiskitFunction(qunova/hivqe-chemistry),
 QiskitFunction(global-data-quantum/quantum-portfolio-optimizer),
 QiskitFunction(algorithmiq/tem),
 QiskitFunction(qedma/qesem),
 QiskitFunction(multiverse/singularity),
 QiskitFunction(ibm/circuit-function),
 QiskitFunction(q-ctrl/optimization-solver),
 QiskitFunction(colibritd/quick-pde),
 QiskitFunction(q-ctrl/performance-management),
 QiskitFunction(kipu-quantum/iskay-quantum-optimizer)]

In [7]:
# load function
singularity = catalog.load("multiverse/singularity")

Realiza los siguientes pasos:

1) Crea el conjunto de datos sintético usando la función [`make_moons`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_moons.html) de [scikit-learn](https://scikit-learn.org/).
2) Sube el conjunto de datos sintético generado al [directorio de datos compartido](https://qiskit.github.io/qiskit-serverless/function_features/experimental/manage_data_directory.html).
3) Crea el clasificador mejorado cuánticamente usando la acción [`create`](https://docs.quantum.ibm.com/api/functions/multiverse-computing-singularity#create).
4) Lista tus clasificadores usando la acción [`list`](https://docs.quantum.ibm.com/api/functions/multiverse-computing-singularity#list).
5) Entrena el clasificador con los datos de entrenamiento usando la acción [`fit`](https://docs.quantum.ibm.com/api/functions/multiverse-computing-singularity#fit).
6) Usa el clasificador entrenado para predecir en los datos de prueba usando la acción [`predict`](https://docs.quantum.ibm.com/api/functions/multiverse-computing-singularity#predict).
7) Elimina el clasificador usando la acción [`delete`](https://docs.quantum.ibm.com/api/functions/multiverse-computing-singularity#delete).
8) Limpia cuando hayas terminado.

**Paso 1.** Importa los módulos necesarios y genera el conjunto de datos sintético, luego divídelo en conjuntos de datos de entrenamiento y prueba.

In [2]:
# import the necessary modules for this example
import os
import tarfile
import numpy as np

# Import the make_moons and the train_test_split functions from scikit-learn
# to create a synthetic dataset and split it into training and test datasets
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

# generate the synthetic dataset
X, y = make_moons(n_samples=10000)

# split the data into training and test datasets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# print the first 10 samples of the training dataset
print("Features:", X_train[:10, :])
print("Targets:", y_train[:10])

Features: [[ 0.84757037 -0.48831433]
 [ 0.98132552  0.19235443]
 [-0.71626723  0.6978261 ]
 [ 1.18957848 -0.48186557]
 [ 0.52118982 -0.37791846]
 [ 0.81115408  0.58483251]
 [ 0.48706462  0.87336593]
 [-0.81880144  0.57407682]
 [ 1.67335408 -0.23932015]
 [ 0.50181306  0.8649761 ]]
Targets: [1 0 0 1 1 0 0 0 1 0]


**Step 2.** Save the labeled training and test datasets on your local disk, and then upload them to the [shared data directory](https://qiskit.github.io/qiskit-serverless/function_features/experimental/manage_data_directory.html).

In [3]:
def make_tarfile(file_path, tar_file_name):
    with tarfile.open(tar_file_name, "w") as tar:
        tar.add(file_path, arcname=os.path.basename(file_path))


# save the training and test datasets on your local disk
np.save("X_train.npy", X_train)
np.save("y_train.npy", y_train)
np.save("X_test.npy", X_test)
np.save("y_test.npy", y_test)

# create tar files for the datasets
make_tarfile("X_train.npy", "X_train.npy.tar")
make_tarfile("y_train.npy", "y_train.npy.tar")
make_tarfile("X_test.npy", "X_test.npy.tar")
make_tarfile("y_test.npy", "y_test.npy.tar")

# upload the datasets to the shared data directory
catalog.file_upload("X_train.npy.tar", singularity)
catalog.file_upload("y_train.npy.tar", singularity)
catalog.file_upload("X_test.npy.tar", singularity)
catalog.file_upload("y_test.npy.tar", singularity)

# view/enlist the uploaded files in the shared data directory
print(catalog.files(singularity))

['X_test.npy.tar', 'X_train.npy.tar', 'y_test.npy.tar', 'y_train.npy.tar']


**Paso 2.** Guarda los conjuntos de datos de entrenamiento y prueba etiquetados en tu disco local y luego súbelos al [directorio de datos compartido](https://qiskit.github.io/qiskit-serverless/function_features/experimental/manage_data_directory.html).

In [4]:
job = singularity.run(
    action="create",
    name="my_classifier",
    num_learners=10,
    learners_types=[
        "DecisionTreeClassifier",
        "KNeighborsClassifier",
    ],
    learners_proportions=[0.5, 0.5],
    learners_options=[{}, {}],
    regularization=0.01,
    weight_update_method="logarithmic",
    sample_scaling=True,
    optimizer_options={"simulator": True},
    voting="soft",
    prob_threshold=0.5,
)

print(job.result())

{'status': 'ok', 'message': 'Classifier created.', 'data': {}, 'metadata': {'resource_usage': {}}}


In [5]:
# list available classifiers using the list action
job = singularity.run(action="list")

print(job.result())

# you can also find your classifiers in the shared data directory with a *.pkl.tar extension
print(catalog.files(singularity))

{'status': 'ok', 'message': 'Classifiers listed.', 'data': {'classifiers': ['my_classifier']}, 'metadata': {'resource_usage': {}}}


['X_test.npy.tar', 'X_train.npy.tar', 'my_classifier.pkl.tar', 'y_test.npy.tar', 'y_train.npy.tar']


**Paso 3.** Crea un clasificador mejorado cuánticamente usando la acción [`create`](https://docs.quantum.ibm.com/api/functions/multiverse-computing-singularity#create).

In [6]:
job = singularity.run(
    action="fit",
    name="my_classifier",
    X="X_train.npy",  # you do not need to specify the tar extension
    y="y_train.npy",  # you do not need to specify the tar extension
)

print(job.result())

{'status': 'ok', 'message': 'Classifier fitted.', 'data': {}, 'metadata': {'resource_usage': {'RUNNING: MAPPING': {'CPU_TIME': 13.655871629714966}, 'RUNNING: WAITING_QPU': {'CPU_TIME': 54.688621282577515}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 56.92286920547485}, 'RUNNING: EXECUTING_QPU': {'QPU_TIME': 57.92738223075867}}}}


**Step 5.** Obtain predictions and probabilities from the quantum-enhanced classifier using the [`predict`](/docs/api/functions/multiverse-computing-singularity#predict) action.

In [7]:
job = singularity.run(
    action="predict",
    name="my_classifier",
    X="X_test.npy",  # you do not need to specify the tar extension
)

result = job.result()

print("Action result status: ", result["status"])
print("Action result message: ", result["message"])
print("Predictions (first five results):", result["data"]["predictions"][:5])
print(
    "Probabilities (first five results):", result["data"]["probabilities"][:5]
)

Action result status:  ok
Action result message:  Classifier predicted.
Predictions (first five results): [0, 0, 1, 0, 1]
Probabilities (first five results): [[1.0, 0.0], [1.0, 0.0], [0.0, 1.0], [1.0, 0.0], [0.0, 1.0]]


**Step 6.** Delete the quantum-enhanced classifier using the [`delete`](/docs/api/functions/multiverse-computing-singularity#delete) action.

In [8]:
job = singularity.run(
    action="delete",
    name="my_classifier",
)

# or you can delete from the shared data directory
# catalog.file_delete("my_classifier.pkl.tar", singularity)

print(job.result())

{'status': 'ok', 'message': 'Classifier deleted.', 'data': {}, 'metadata': {'resource_usage': {}}}


**Step 7.** Clean up local and shared data directories.

In [9]:
# delete the numpy files from your local disk
os.remove("X_train.npy")
os.remove("y_train.npy")
os.remove("X_test.npy")
os.remove("y_test.npy")

# delete the tar files from your local disk
os.remove("X_train.npy.tar")
os.remove("y_train.npy.tar")
os.remove("X_test.npy.tar")
os.remove("y_test.npy.tar")

# delete the tar files from the shared data
catalog.file_delete("X_train.npy.tar", singularity)
catalog.file_delete("y_train.npy.tar", singularity)
catalog.file_delete("X_test.npy.tar", singularity)
catalog.file_delete("y_test.npy.tar", singularity)

'Requested file was deleted.'

### `create_fit_predict` example

The following example demonstrates the [`create_fit_predict`](/docs/api/functions/multiverse-computing-singularity#create_fit_predict) action.

In [10]:
# Import QiskitFunctionsCatalog to load the
# "Singularity Machine Learning - Classification" function by Multiverse Computing
from qiskit_ibm_catalog import QiskitFunctionsCatalog

# Import the make_moons and the train_test_split functions from scikit-learn
# to create a synthetic dataset and split it into training and test datasets
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

# authentication
# If you have not previously saved your credentials, follow instructions at
# /docs/guides/functions
# to authenticate with your API key.
catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# load "Singularity Machine Learning - Classification" function by Multiverse Computing
singularity = catalog.load("multiverse/singularity")

# generate the synthetic dataset
X, y = make_moons(n_samples=1000)

# split the data into training and test datasets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

job = singularity.run(
    action="create_fit_predict",
    num_learners=10,
    regularization=0.01,
    optimizer_options={"simulator": True},
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    options={"save": False},
)

# get job status and result
status = job.status()
result = job.result()

print("Job status: ", status)
print("Action result status: ", result["status"])
print("Action result message: ", result["message"])
print("Predictions (first five results): ", result["data"]["predictions"][:5])
print(
    "Probabilities (first five results): ",
    result["data"]["probabilities"][:5],
)
print("Usage metadata: ", result["metadata"]["resource_usage"])

Job status:  QUEUED
Action result status:  ok
Action result message:  Classifier created, fitted, and predicted.
Predictions (first five results):  [0, 0, 1, 0, 0]
Probabilities (first five results):  [[0.87119766531518, 0.1288023346848197], [0.87119766531518, 0.1288023346848197], [0.24470328446479797, 0.7552967155352032], [0.820524432250189, 0.17947556774981072], [0.6847610293419495, 0.31523897065805173]]
Usage metadata:  {'RUNNING: MAPPING': {'CPU_TIME': 10.967791318893433}, 'RUNNING: WAITING_QPU': {'CPU_TIME': 59.91712307929993}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 59.097386837005615}, 'RUNNING: EXECUTING_QPU': {'QPU_TIME': 56.93338203430176}}


**Paso 4.** Entrena el clasificador mejorado cuánticamente usando la acción [`fit`](https://docs.quantum.ibm.com/api/functions/multiverse-computing-singularity#fit).